# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations (30-day readmissions) and role-based column masking using DuckDB.

In [ ]:
!pip install duckdb pandas -q

In [ ]:
import duckdb

conn = duckdb.connect(database=':memory:')
print("DuckDB connected.")

In [ ]:
# Setup Mock Data
conn.execute("""
    CREATE TABLE omop_condition_occurrence (
        condition_occurrence_id VARCHAR,
        person_id VARCHAR,
        condition_type_concept_id INT,
        condition_start_date DATE,
        condition_end_date DATE
    );
""")

conn.execute("""
    INSERT INTO omop_condition_occurrence VALUES 
    ('ENC1', 'P123', 32020, '2023-10-01', '2023-10-05'), -- Discharge on 10-05
    ('ENC2', 'P123', 32020, '2023-10-20', NULL),         -- Readmission within 30 days
    ('ENC3', 'P456', 32020, '2023-09-01', '2023-09-10'), -- Discharge
    ('ENC4', 'P456', 32020, '2023-11-01', NULL);         -- Readmission outside 30 days
""")

In [ ]:
# Simulate dbt Gold Mart: 30-day Readmissions
query_readmissions = """
WITH discharges AS (
    SELECT 
        person_id,
        condition_occurrence_id AS discharge_encounter_id,
        condition_end_date AS discharge_date
    FROM omop_condition_occurrence
    WHERE condition_end_date IS NOT NULL
),
admissions AS (
    SELECT
        person_id,
        condition_occurrence_id AS admission_encounter_id,
        condition_start_date AS admission_date
    FROM omop_condition_occurrence
)
SELECT
    d.person_id,
    d.discharge_encounter_id,
    d.discharge_date,
    a.admission_encounter_id,
    a.admission_date,
    DATE_DIFF('day', CAST(d.discharge_date AS TIMESTAMP), CAST(a.admission_date AS TIMESTAMP)) AS days_to_readmission
FROM discharges d
JOIN admissions a 
  ON d.person_id = a.person_id
 AND a.admission_date > d.discharge_date
 AND DATE_DIFF('day', CAST(d.discharge_date AS TIMESTAMP), CAST(a.admission_date AS TIMESTAMP)) <= 30
"""

print("Gold Mart: 30-Day Readmissions")
print(conn.execute(query_readmissions).df())

In [ ]:
# Simulate Dynamic Data Masking
conn.execute("""
    CREATE TABLE patient_demographics (
        person_id VARCHAR,
        ssn VARCHAR,
        dob DATE
    );
    INSERT INTO patient_demographics VALUES ('P123', '123-45-6789', '1980-05-15');
""")

def query_masked_data(role):
    if role == 'PHI_VIEWER':
        return conn.execute("SELECT person_id, ssn, dob FROM patient_demographics").df()
    else:
        return conn.execute("""
            SELECT 
                person_id, 
                '***-**-****' AS ssn, 
                DATE_TRUNC('year', dob) AS dob_masked 
            FROM patient_demographics
        """).df()

print("\nData view for role: PHI_VIEWER")
print(query_masked_data('PHI_VIEWER'))

print("\nData view for role: ANALYST (Masked)")
print(query_masked_data('ANALYST'))